# Business Units

In [1]:
import pandas as pd

# Load all summaries
scope1_by_bu = pd.read_csv("scope1_summary_by_bu.csv")
scope1_by_category = pd.read_csv("scope1_summary_by_category.csv")
scope2_by_bu = pd.read_csv("scope2_summary_by_bu.csv")
scope2_by_category = pd.read_csv("scope2_summary_by_category.csv")
scope3_by_bu = pd.read_csv("scope3_summary_by_bu.csv")
scope3_by_category = pd.read_csv("scope3_summary_by_category.csv")

# Display heads
print("Scope 1 by BU")
display(scope1_by_bu.head())

print("Scope 1 by Category")
display(scope1_by_category.head())

print("Scope 2 by BU")
display(scope2_by_bu.head())

print("Scope 2 by Category")
display(scope2_by_category.head())

print("Scope 3 by BU")
display(scope3_by_bu.head())

print("Scope 3 by Category")
display(scope3_by_category.head())

Scope 1 by BU


,Scope,Business Unit,Category,Emissions(tCO2e)
0,Scope 1,BU1,Biodiesel,1.105566
1,Scope 1,BU10,Natural Gas,4496.805034
2,Scope 1,BU10,Biodiesel,1.440586
3,Scope 1,BU11,Biodiesel,0.361822
4,Scope 1,BU2,Fertilizer,9381.376390


Scope 1 by Category


,Scope,Category,Emissions(tCO2e)
0,Scope 1,Biodiesel,1.675134e+06
1,Scope 1,Fertilizer,3.216297e+04
2,Scope 1,Natural Gas,8.271387e+03


Scope 2 by BU


...

,Category,Emissions (tCO2e)
0,Electricity (Scope 2),2525.463637


Scope 3 by BU


,BU,Subcategory,Category,Emissions (tCO2e)
0,BU1,CGK - PKU,Business Travel,13.343712
1,BU1,PKU - CGK,Business Travel,9.690964
2,BU1,PKU - YIA,Business Travel,0.852869
3,BU1,PKU - HLP,Business Travel,0.384352
4,BU1,SMQ - PKU,Business Travel,0.318383


Scope 3 by Category


,Subcategory,Category,Emissions (tCO2e)
0,Coffee Bean,Purchased Goods,485958.182600
1,Sesame,Purchased Goods,312897.859200
2,Cocoa Powder,Purchased Goods,239822.583908
3,Sesame,Waste,238447.266750
4,Onion,Waste,100500.000000


In [2]:
# Combine BU-level summaries
bu_scope1 = scope1_by_bu.rename(columns={"Business Unit": "BU"})
bu_scope1["Scope"] = "Scope 1"

bu_scope2 = scope2_by_bu.rename(columns={"Emissions (tCO2e)": "Emissions_tCO2e"})
bu_scope2["Scope"] = "Scope 2"

bu_scope3 = scope3_by_bu.groupby("BU")["Emissions (tCO2e)"].sum().reset_index()
bu_scope3["Scope"] = "Scope 3"
bu_scope3 = bu_scope3.rename(columns={"Emissions (tCO2e)": "Emissions_tCO2e"})

# Standardize Scope 1 column name
bu_scope1 = bu_scope1.rename(columns={"Emissions(tCO2e)": "Emissions_tCO2e"})

# Combine all scopes
bu = pd.concat([bu_scope1[["BU", "Scope", "Emissions_tCO2e"]],
                bu_scope2[["BU", "Scope", "Emissions_tCO2e"]],
                bu_scope3[["BU", "Scope", "Emissions_tCO2e"]]],
               ignore_index=True)


In [6]:
# --- Standardize Scope 1 column ---
scope1_by_bu = scope1_by_bu.rename(columns={
    "Business Unit": "BU",
    "Emissions(tCO2e)": "Emissions (tCO2e)"
})

# --- Standardize Scope 2 column ---
scope2_by_bu = scope2_by_bu.rename(columns={
    "Business Unit": "BU",
    "Emissions(tCO2e)": "Emissions (tCO2e)"
})

# --- Assign Scope labels ---
scope1_by_bu["Scope"] = "Scope 1"
scope2_by_bu["Scope"] = "Scope 2"

# --- Clean Scope 3: Remove subcategory splits ---
scope3_by_bu_clean = scope3_by_bu.groupby(["BU", "Category"], as_index=False)["Emissions (tCO2e)"].sum()
scope3_by_bu_clean["Scope"] = "Scope 3"

# --- Combine all scopes ---
bu_category_df = pd.concat([
    scope1_by_bu[["Scope", "BU", "Category", "Emissions (tCO2e)"]],
    scope2_by_bu[["Scope", "BU", "Category", "Emissions (tCO2e)"]],
    scope3_by_bu_clean[["Scope", "BU", "Category", "Emissions (tCO2e)"]]
], ignore_index=True)

In [7]:
bu_category_df

,Scope,BU,Category,Emissions (tCO2e)
0,Scope 1,BU1,Biodiesel,1.105566e+00
1,Scope 1,BU10,Natural Gas,4.496805e+03
2,Scope 1,BU10,Biodiesel,1.440586e+00
3,Scope 1,BU11,Biodiesel,3.618216e-01
4,Scope 1,BU2,Fertilizer,9.381376e+03
5,Scope 1,BU2,Biodiesel,2.683510e+00
6,Scope 1,BU3,Biodiesel,1.675104e+06
7,Scope 1,BU3,Fertilizer,1.779281e+04
8,Scope 1,BU4,Biodiesel,3.916384e+00
9,Scope 1,BU5,Fertilizer,3.019153e+03


In [8]:
import plotly.express as px
import pandas as pd

# Ensure Scope order is explicitly defined
scope_order = ["Scope 1", "Scope 2", "Scope 3"]

# Categorize and sort
scope_distribution["Scope"] = pd.Categorical(scope_distribution["Scope"], categories=scope_order, ordered=True)

# Plot horizontal bar chart with correct top-to-bottom sorting
fig_bar = px.bar(
    scope_distribution,
    x="Emissions (tCO2e)",
    y="Scope",
    orientation="h",
    title="Distribution of Emissions by Scope",
    text="Emissions (tCO2e)",
    labels={"Emissions (tCO2e)": "Emissions (tCO₂e)"}
)

fig_bar.update_layout(
    yaxis=dict(categoryorder="array", categoryarray=scope_order[::-1]),  # reverse for top-down
    xaxis_title="Emissions (tCO₂e)",
    yaxis_title=""
)

fig_bar.update_traces(texttemplate="%{text:.2s}", textposition="outside")
fig_bar.show()
fig_bar.write_html("Images/emissions_by_scope.html")

NameError: name 'scope_distribution' is not defined

In [12]:
import plotly.express as px

# Compute total emissions per BU to determine sort order
bu_order = (
    bu_category_df.groupby("BU", as_index=False)["Emissions (tCO2e)"]
    .sum()
    .sort_values("Emissions (tCO2e)", ascending=True)
)

# Merge order into original df for correct sorting
bu_category_df_sorted = bu_category_df.copy()
bu_category_df_sorted["BU"] = pd.Categorical(
    bu_category_df_sorted["BU"],
    categories=bu_order["BU"],
    ordered=True
)

# Plot horizontal stacked bar chart sorted by total emissions per BU
fig_bu_cat = px.bar(
    bu_category_df_sorted,
    y="BU",
    x="Emissions (tCO2e)",
    color="Category",
    orientation="h",
    title="Emissions by Business Unit Segmented by Category",
    text_auto=".2s",
    labels={"Emissions (tCO2e)": "Emissions (tCO₂e)"}
)

fig_bu_cat.update_layout(barmode="stack")
fig_bu_cat.show()

In [15]:
import plotly.express as px

# Calculate total emissions per BU for sorting
bu_order = (
    bu_category_df.groupby("BU")["Emissions (tCO2e)"]
    .sum()
    .sort_values(ascending=False)
    .index.tolist()
)

# Plot with custom BU order
fig_bu_cat = px.bar(
    bu_category_df,
    y="BU",
    x="Emissions (tCO2e)",
    color="Category",
    orientation="h",
    category_orders={"BU": bu_order},
    title="Emissions by Business Unit Segmented by Category",
    text_auto=".2s",
    labels={"Emissions (tCO2e)": "Emissions (tCO₂e)"}
)

fig_bu_cat.update_layout(barmode="stack")
fig_bu_cat.show()

In [13]:
total_emissions = bu_category_df["Emissions (tCO2e)"].sum()
print(f"Total GHG Emissions: {total_emissions:,.2f} tCO₂e")

Total GHG Emissions: 3,443,446.72 tCO₂e


In [16]:
import pandas as pd
import plotly.express as px

# --- Load Scope 1 Summary CSVs ---
scope1_by_bu = pd.read_csv("scope1_summary_by_bu.csv")
scope1_by_category = pd.read_csv("scope1_summary_by_category.csv")

# --- Horizontal Bar Chart: Scope 1 Emissions by Business Unit ---
fig_scope1_bu = px.bar(
    scope1_by_bu.sort_values("Emissions(tCO2e)", ascending=True),
    y="Business Unit",
    x="Emissions(tCO2e)",
    orientation="h",
    text_auto=".2s",
    title="Scope 1 Emissions by Business Unit",
    labels={"Emissions(tCO2e)": "Emissions (tCO₂e)"}
)
fig_scope1_bu.update_layout(yaxis=dict(categoryorder='total ascending'))
fig_scope1_bu.show()

# --- Horizontal Bar Chart: Scope 1 Emissions by Category ---
fig_scope1_cat = px.bar(
    scope1_by_category.sort_values("Emissions(tCO2e)", ascending=True),
    y="Category",
    x="Emissions(tCO2e)",
    orientation="h",
    text_auto=".2s",
    title="Scope 1 Emissions by Category",
    labels={"Emissions(tCO2e)": "Emissions (tCO₂e)"}
)
fig_scope1_cat.update_layout(yaxis=dict(categoryorder='total ascending'))
fig_scope1_cat.show()

In [18]:
import plotly.express as px

# --- Filter Scope 3 only ---
scope3_only = bu_category_df[bu_category_df["Scope"] == "Scope 3"]

# 1. Scope 3 Emissions by Category (Vertical Bar)
fig_scope3_cat = px.bar(
    scope3_only.groupby("Category", as_index=False)["Emissions (tCO2e)"].sum().sort_values("Emissions (tCO2e)", ascending=False),
    x="Category",
    y="Emissions (tCO2e)",
    title="Scope 3 Emissions by Category",
    text_auto=".2s",
    labels={"Emissions (tCO2e)": "Emissions (tCO₂e)"}
)
fig_scope3_cat.show()

# 2. Scope 3 Emissions by Business Unit (Vertical Bar)
fig_scope3_bu = px.bar(
    scope3_only.groupby("BU", as_index=False)["Emissions (tCO2e)"].sum().sort_values("Emissions (tCO2e)", ascending=False),
    x="BU",
    y="Emissions (tCO2e)",
    title="Scope 3 Emissions by Business Unit",
    text_auto=".2s",
    labels={"Emissions (tCO2e)": "Emissions (tCO₂e)"}
)
fig_scope3_bu.show()

# 3. Scope 3 Emissions by BU and Category (Segmented)
fig_scope3_bu_cat = px.bar(
    scope3_only.sort_values("Emissions (tCO2e)", ascending=False),
    x="BU",
    y="Emissions (tCO2e)",
    color="Category",
    title="Scope 3 Emissions by Business Unit and Category",
    text_auto=".2s",
    labels={"Emissions (tCO2e)": "Emissions (tCO₂e)"}
)
fig_scope3_bu_cat.update_layout(barmode="stack")
fig_scope3_bu_cat.show()

In [ ]:
import plotly.express as px

# --- Group total emissions by Scope ---
scope_distribution = bu_category_df.groupby("Scope", as_index=False)["Emissions (tCO2e)"].sum()

# --- PIE Chart ---
fig_pie = px.pie(
    scope_distribution,
    names="Scope",
    values="Emissions (tCO2e)",
    title="Distribution of Emissions by Scope",
    hole=0.4  # donut style
)
fig_pie.show()

import plotly.express as px

# --- Group total emissions by Scope and Category ---
scope_category_summary = bu_category_df.groupby(["Scope", "Category"], as_index=False)["Emissions (tCO2e)"].sum()

# --- Stacked BAR Chart: Emissions by Scope segmented by Category ---
fig = px.bar(
    scope_category_summary,
    x="Scope",
    y="Emissions (tCO2e)",
    color="Category",
    title="Emissions by Scope and Category",
    text_auto=".2s"
)

fig.update_layout(barmode="stack")
fig.show()

In [ ]:
import plotly.express as px

# Pie chart: Scope 1 by BU
fig_scope1 = px.pie(
    bu[bu["Scope"] == "Scope 1"],
    values="Emissions_tCO2e",
    names="BU",
    title="Scope 1 Emissions by BU"
)
fig_scope1.show()

# Pie chart: Scope 2 by BU
fig_scope2 = px.pie(
    bu[bu["Scope"] == "Scope 2"],
    values="Emissions_tCO2e",
    names="BU",
    title="Scope 2 Emissions by BU"
)
fig_scope2.show()

# Pie chart: Scope 3 by BU
fig_scope3 = px.pie(
    bu[bu["Scope"] == "Scope 3"],
    values="Emissions_tCO2e",
    names="BU",
    title="Scope 3 Emissions by BU"
)
fig_scope3.show()

In [ ]:
# Standardize column names
scope1_by_category = scope1_by_category.rename(columns={"Emissions(tCO2e)": "Emissions (tCO2e)"})
scope2_by_category = scope2_by_category.rename(columns={"Emissions (tCO2e)": "Emissions (tCO2e)"})
scope3_by_category = scope3_by_category.rename(columns={"Emissions (tCO2e)": "Emissions (tCO2e)"})

# Group Scope 3 by Category (it has Subcategories)
scope3_grouped = (
    scope3_by_category
    .groupby("Category")["Emissions (tCO2e)"]
    .sum()
    .reset_index()
)

# Combine all
category = pd.concat([
    scope1_by_category.assign(Scope="Scope 1"),
    scope2_by_category.assign(Scope="Scope 2"),
    scope3_grouped.assign(Scope="Scope 3")
], ignore_index=True)

# Reorder columns
category = category[["Scope", "Category", "Emissions (tCO2e)"]]

# Display result
display(category)

In [ ]:
bu_category_df

In [ ]:
bu_category_df

In [ ]:


# Add Scope column
scope1_by_bu["Scope"] = "Scope 1"
scope2_by_bu["Scope"] = "Scope 2"
scope3_by_bu["Scope"] = "Scope 3"

# Combine them
bu_category_df = pd.concat([scope1_by_bu, scope2_by_bu, scope3_by_bu], ignore_index=True)
# Prioritize 'Business Unit' if available, otherwise use 'BU'
bu_category_df["BU"] = bu_category_df["Business Unit"].combine_first(bu_category_df["BU"])

# Drop the redundant 'Business Unit' column
bu_category_df = bu_category_df.drop(columns=["Business Unit"])


# Create the stacked bar chart
import plotly.express as px

fig = px.bar(
    bu_category_df,
    x="BU",
    y="Emissions (tCO2e)",
    color="Category",
    title="Emissions by BU segmented by Category (All Scopes)",
    labels={"Emissions (tCO2e)": "Emissions (tCO₂e)"},
    text_auto=".2s"
)
fig.update_layout(barmode='stack')
fig.show()

In [ ]:
scope1_by_bu